In [1]:
# 다운로드 받는 코드
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews")
path = path.replace('\\', '/')

review_df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))
movie_df = pd.read_csv(os.path.join(path, os.listdir(path)[1]))

print("Path to dataset files:", path)
print(movie_df.shape)
print(review_df.shape)

/home/shin/anaconda3/envs/rec-modeling-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/shin/.cache/kagglehub/datasets/andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews/versions/4
(143258, 16)
(1444963, 11)


## Data train & test split

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

review_df.drop_duplicates(inplace=True)
review_df.reset_index(drop=True, inplace=True)

selected_movie_ids = review_df['id'].value_counts()[review_df['id'].value_counts()>=56].index
selected_review_df = review_df[review_df['id'].isin(selected_movie_ids)]
selected_review_df.reset_index(drop=True, inplace=True)

selected_reviewer_names = selected_review_df['criticName'].value_counts()[selected_review_df['criticName'].value_counts()>=4].index
selected_review_df = selected_review_df[selected_review_df['criticName'].isin(selected_reviewer_names)]
selected_review_df.reset_index(drop=True, inplace=True)

X_train, X_test, Y_train, Y_test = train_test_split(selected_review_df.drop('reviewState', axis=1), selected_review_df['reviewState'], stratify=selected_review_df.criticName, test_size=0.1)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, stratify=X_train.criticName, test_size=0.25)

In [3]:
user2idx = {review: idx for idx, review in enumerate(selected_review_df['criticName'].unique())}
item2idx = {movie: idx for idx, movie in enumerate(selected_review_df['id'].unique())}

In [4]:
from tqdm.auto import tqdm

n = len(user2idx)
m = len(item2idx)
user_item_matrix = np.full((n, m), 0.5)

for idx, row in tqdm(X_train.iterrows(), total=len(X_train)):
    user_item_matrix[user2idx[row['criticName']], item2idx[row['id']]] = float(Y_train.loc[idx] == 'fresh')

100%|██████████| 619254/619254 [00:13<00:00, 44337.17it/s]


In [5]:
user_item_matrix

array([[1. , 0.5, 0.5, ..., 1. , 0.5, 1. ],
       [0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5],
       ...,
       [0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5]])

## Modeling

### SVD

In [6]:
import scipy

K = 64

u, s, vh = scipy.sparse.linalg.svds(user_item_matrix, k=K)
pred_user_item_matrix = np.dot(np.dot(u, np.diag(s)), vh)

In [7]:
pred_user_item_matrix

array([[0.5997594 , 0.48509264, 0.43930877, ..., 0.87993977, 0.52260867,
        0.83394372],
       [0.4960688 , 0.50606164, 0.49808531, ..., 0.53017942, 0.47932297,
        0.50426418],
       [0.50516109, 0.50690336, 0.48600139, ..., 0.52524275, 0.49590914,
        0.48715863],
       ...,
       [0.5003847 , 0.50008104, 0.49850934, ..., 0.49890081, 0.50255683,
        0.49731436],
       [0.49909763, 0.49915085, 0.49993484, ..., 0.50152135, 0.50429681,
        0.49973787],
       [0.4993993 , 0.50063726, 0.49879953, ..., 0.49939132, 0.50256664,
        0.50013086]])

#### validate & test

In [8]:
X_val.head()

,id,reviewId,creationDate,criticName,isTopCritic,originalScore,publicatioName,reviewText,scoreSentiment,reviewUrl
842145,blonde,102739056,2022-10-04,Tim Brennan,False,NaN,About Boulder,Ana de Armas is in nearly every scene&#44; and...,NEGATIVE,https://aboutboulder.com/blog/norma-jeane/
391002,the_wolf_of_snow_hollow,2732574,2020-10-09,Sean P. Means,False,3/4,Salt Lake Tribune,"As in his Sundance award-winning short ""Thunde...",POSITIVE,https://www.sltrib.com/artsliving/2020/10/08/r...
829999,2012,1854733,2009-11-11,Josh Bell,False,2.5/5,Las Vegas Weekly,"It's an exhausting, clumsy mess, and its momen...",NEGATIVE,http://www.lasvegasweekly.com/news/2009/nov/11...
908638,the_children_act,2512587,2018-09-23,Tom Santilli,False,C,AXS.com,"There's an artificiality to it, from the set p...",NEGATIVE,https://www.axs.com/reviews-the-house-with-a-c...
506866,buried,1972866,2011-04-04,Norman Wilner,False,4/5,NOW Toronto,"Reynolds is riveting, and it's possible to app...",POSITIVE,NaN


In [9]:
def pred(df):
    preds = []
    for idx, row in df.iterrows():
        pred = pred_user_item_matrix[user2idx[row['criticName']], item2idx[row['id']]]
        preds.append(float(pred > 0.5))
    return np.array(preds)

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def print_metric(predicts, ground_truth):
    metrics = {
        'accuracy': accuracy_score(ground_truth, predicts),
        'precision': precision_score(ground_truth, predicts),
        'recall': recall_score(ground_truth, predicts),
        'f1': f1_score(ground_truth, predicts),
        'auc_roc': roc_auc_score(ground_truth, predicts)
    }
    for metric, value in metrics.items():
        print(f'{metric}: {value}')

In [20]:
pred_val = pred(X_val)
Y_val_float = (Y_val.values == "fresh").astype(float)

In [26]:
print_metric(pred_val, Y_val_float)

accuracy: 0.7251658035355273
precision: 0.7880726900435117
recall: 0.8045758418198843
f1: 0.7962387624407817
auc_roc: 0.6851930074332773


In [27]:
pred_test = pred(X_test)
Y_test_float = (Y_test.values == "fresh").astype(float)

print_metric(pred_test, Y_test_float)

accuracy: 0.7233982254583506
precision: 0.7838326611612573
recall: 0.8049337672155935
f1: 0.7942430876510176
auc_roc: 0.6838787168548189


### K=32

In [28]:
import scipy

K = 32

u, s, vh = scipy.sparse.linalg.svds(user_item_matrix, k=K)
pred_user_item_matrix = np.dot(np.dot(u, np.diag(s)), vh)

In [29]:
pred_val = pred(X_val)
Y_val_float = (Y_val.values == "fresh").astype(float)

In [30]:
print_metric(pred_val, Y_val_float)

accuracy: 0.7501489688449222
precision: 0.7987812064530889
recall: 0.8363178409924003
f1: 0.8171186633003319
auc_roc: 0.7067739644861786


In [31]:
pred_test = pred(X_test)
Y_test_float = (Y_test.values == "fresh").astype(float)

print_metric(pred_test, Y_test_float)

accuracy: 0.7473785180179198
precision: 0.794573037222396
recall: 0.8349768267429247
f1: 0.8142740371516035
auc_roc: 0.704920443407713


### K=128

In [32]:
import scipy

K = 128

u, s, vh = scipy.sparse.linalg.svds(user_item_matrix, k=K)
pred_user_item_matrix = np.dot(np.dot(u, np.diag(s)), vh)

pred_val = pred(X_val)
Y_val_float = (Y_val.values == "fresh").astype(float)

print_metric(pred_val, Y_val_float)

accuracy: 0.6919033616091542
precision: 0.7738072385227626
recall: 0.7607482198204214
f1: 0.7672221632523086
auc_roc: 0.657248782170305


In [33]:
pred_test = pred(X_test)
Y_test_float = (Y_test.values == "fresh").astype(float)

print_metric(pred_test, Y_test_float)

accuracy: 0.6916788384818294
precision: 0.769233313488126
recall: 0.7644545245373566
f1: 0.7668364739436504
auc_roc: 0.6564051493738052
